# Chapter 3 — The Reason–Act–Observe Loop

Hands-on lab: hand-write a ReAct trace for a small task, then rewrite it as a
CodeAct trace, per `agent_code_execution_study_guide.md` Chapter 3's hands-on
direction.

Task: which of `a.txt`, `b.txt`, `c.txt` holds the largest number, and what is
that number plus 10? Both traces below run through a *real* loop against a
real (in-memory) environment — only the model's output is scripted (see
[`../code/react_vs_codeact.py`](../code/react_vs_codeact.py) for why that
distinction matters and is preserved throughout).

In [1]:
import sys
sys.path.insert(0, "../code")

from react_vs_codeact import (
    TASK_PROMPT,
    REACT_SCRIPT,
    ScriptedReActModel,
    run_react_loop,
    CODEACT_THOUGHT,
    CODEACT_CODE,
    run_codeact_loop,
    read_file,
    HISTORICAL_TIMELINE,
    render_timeline,
)

print(TASK_PROMPT)

Which of a.txt, b.txt, c.txt contains the largest number, and what is that number plus 10?


## 1. The ReAct trace

`REACT_SCRIPT` is hand-written: four `ReActStep`s, each pairing a `Thought`
with an `Action` (or, on the last step, a `Finish`). `ScriptedReActModel`
plays these back one at a time — standing in for a real model call, which
starts in Chapter 5 — but `run_react_loop` is a real loop: it really calls
`read_file` for each action and really threads the result back into context
as the next `Observation`, exactly per the "Thought / Action / Observation"
anatomy ReAct introduced.

In [2]:
react_transcript = run_react_loop(ScriptedReActModel(REACT_SCRIPT))
for kind, text in react_transcript:
    print(f"{kind}: {text}")

n_actions = sum(1 for k, _ in react_transcript if k == "Action")
print(f"\n{n_actions} actions, {len(react_transcript)} trace entries")

Thought: I need each file's value before I can compare them. Start with a.txt.
Action: read_file('a.txt')
Observation: 42
Thought: a.txt is 42. Now check b.txt.
Action: read_file('b.txt')
Observation: 17
Thought: b.txt is 17, smaller than 42 so far. Now check c.txt.
Action: read_file('c.txt')
Observation: 8
Thought: c.txt is 8. The largest value seen is 42, from a.txt. 42 + 10 = 52.
Finish: a.txt has the largest number (42); 42 + 10 = 52.

3 actions, 11 trace entries


## 2. The same task, rewritten as CodeAct

One `Thought`, one code block that does all three reads and the comparison in
Python's own control flow, then a `Finish`. `run_codeact_loop` really
`exec()`s `CODEACT_CODE` and captures its real stdout as the `Observation` —
nothing about the arithmetic or comparison is hand-computed ahead of time.

In [3]:
print(CODEACT_CODE)

values = {f: int(read_file(f)) for f in ["a.txt", "b.txt", "c.txt"]}
largest_file = max(values, key=values.get)
result = values[largest_file] + 10
print(f"{largest_file} has the largest number ({values[largest_file]}); "
      f"{values[largest_file]} + 10 = {result}")



In [4]:
codeact_transcript = run_codeact_loop(CODEACT_THOUGHT, CODEACT_CODE, {"read_file": read_file})
for kind, text in codeact_transcript:
    print(f"{kind}: {text}")

n_actions = sum(1 for k, _ in codeact_transcript if k.startswith("Action"))
print(f"\n{n_actions} action, {len(codeact_transcript)} trace entries")

Thought: I'll read all three files, compare them, and compute the answer in one action.
Action (code): values = {f: int(read_file(f)) for f in ["a.txt", "b.txt", "c.txt"]}
largest_file = max(values, key=values.get)
result = values[largest_file] + 10
print(f"{largest_file} has the largest number ({values[largest_file]}); "
      f"{values[largest_file]} + 10 = {result}")
Observation: a.txt has the largest number (42); 42 + 10 = 52
Finish: a.txt has the largest number (42); 42 + 10 = 52

1 action, 4 trace entries


## 3. Same answer, different shape

Both traces land on the identical, correctly computed answer
(`a.txt has the largest number (42); 42 + 10 = 52`) — the rewrite didn't
change *what* gets computed, only *how many actions* it took to get there:
ReAct needed 3 actions (one `read_file` per file) interleaved with 4
thoughts; CodeAct needed 1 action, because the comparison logic that ReAct
had to spread across 3 separate Thought steps is just ordinary Python control
flow (a dict comprehension + `max`) inside CodeAct's single code block. This
is the "From ReAct to CodeAct" shift the chapter's subtopic names: the
interleaving pattern (reason, act, observe, repeat) doesn't change — only the
action's *shape* does, from one-tool-call-per-turn to one-code-block-that-
composes-many-operations.

## 4. Related lineage

The shift didn't happen in one step. `HISTORICAL_TIMELINE` in
`react_vs_codeact.py` records four papers in order, each verified (title,
authors, submission date) directly against their arXiv abstract pages rather
than recalled from memory:

In [5]:
print(render_timeline())

2022-10-06  ReAct
           Yao, Zhao, Yu, Du, Shafran, Narasimhan, Cao — arXiv:2210.03629
           Interleaves free-text Thought/Action/Observation steps; the action is a call into a small, fixed tool set, expressed and parsed as text.

2022-11-18  PAL
           Gao, Madaan, Zhou, Alon, Liu, Yang, Callan, Neubig — arXiv:2211.10435
           Offloads the reasoning chain's actual computation to a generated Python program run by an external interpreter, instead of having the LLM carry out arithmetic/logic itself in text.

2023-02-09  Toolformer
           Schick, Dwivedi-Yu, Dessi, Raileanu, Lomeli, Zettlemoyer, Cancedda, Scialom — arXiv:2302.04761
           Trains the model itself, via self-supervised fine-tuning, to decide when and how to call tools inline during generation, rather than relying on prompting to elicit tool calls.

2024-02-01  CodeAct
           Wang, Chen, Yuan, Zhang, Li, Peng, Ji — arXiv:2402.01030
           Proposes executable code as a unified action space fo

Reading the progression: ReAct establishes the interleaved loop with
text-based actions. PAL (one month later) shows that offloading *computation*
specifically to a real Python interpreter beats doing arithmetic/logic in the
model's own text — but PAL is a single generate-then-execute step, not an
iterated loop. Toolformer explores training the tool-use decision into the
model itself rather than eliciting it via prompting. CodeAct (2024) is the
synthesis: keep ReAct's iterated loop, but make the action *itself* an
arbitrary, composable Python program the way PAL showed was more reliable for
computation — which is exactly the ReAct-trace-to-CodeAct-trace rewrite this
notebook just did by hand.